In [4]:
import torch
from shap_e.models.download import load_model, load_config
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config

print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")

Torch version: 2.10.0+cu128
CUDA available: True
Device: Tesla T4


In [5]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [6]:
from shap_e.models.download import load_model, load_config

xm = load_model('transmitter', device=device)
model = load_model('text300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))

/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:31: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:43: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:61: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd
/usr/local/lib/python3.12/dist-packages/shap_e/models/nn/checkpoint.py:86: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @custom_bwd


  0%|          | 0.00/1.78G [00:00<?, ?iB/s]

100%|████████████████████████████████████████| 890M/890M [00:04<00:00, 191MiB/s]


  0%|          | 0.00/1.26G [00:00<?, ?iB/s]

In [15]:
from shap_e.diffusion.sample import sample_latents

batch_size = 1
guidance_scale = 15.0
prompt = "a medieval helmet"

latents = sample_latents(
    batch_size=batch_size,
    model=model,
    diffusion=diffusion,
    guidance_scale=guidance_scale,
    model_kwargs=dict(texts=[prompt] * batch_size),
    progress=True,
    clip_denoised=True,
    use_fp16=True,
    use_karras=True,
    karras_steps=64,
    sigma_min=1e-3,
    sigma_max=160,
    s_churn=0,
)

print(f"Latents shape: {latents.shape}")
print(f"Latents dtype: {latents.dtype}")

  0%|          | 0/64 [00:00<?, ?it/s]

Latents shape: torch.Size([1, 1048576])
Latents dtype: torch.float32


In [16]:
from shap_e.util.notebooks import decode_latent_mesh
import os

output_dir = '/content/shape-e-model-generator/outputs'
os.makedirs(output_dir, exist_ok=True)

for i, latent in enumerate(latents):
    mesh = decode_latent_mesh(xm, latent).tri_mesh()

    output_path = os.path.join(output_dir, f'medieval_helmet_{i}.obj')
    with open(output_path, 'w') as f:
        mesh.write_obj(f)

    print(f"Saved: {output_path}")

Saved: /content/shape-e-model-generator/outputs/medieval_helmet_0.obj


In [13]:
%cd /content/shape-e-model-generator
!git config user.email "shreyans.sahu07@gmail.com"
!git config user.name "shreyans-gits"

/content/shape-e-model-generator


In [14]:
from google.colab import userdata
token = userdata.get('GITHUB_PAT')

!git add outputs/wooden_chair_0.obj
!git commit -m "add first generated 3D model - wooden chair"
!git push https://{token}@github.com/shreyans-gits/shape-e-model-generator.git

[main 29346cd] add first generated 3D model - wooden chair
 1 file changed, 61690 insertions(+)
 create mode 100644 outputs/wooden_chair_0.obj
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (4/4), 1021.40 KiB | 3.78 MiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/shreyans-gits/shape-e-model-generator.git
   b847643..29346cd  main -> main
